# Approach 2 — MLP Attribute Extraction Attack

## The contrast with Approach 1

Approach 1 required manual effort : the attacker crafted probes by hand and refined them iteratively.

Approach 2 automates the attack.

The attacker trains small neural networks on a labeled public dataset :  
embeddings as input, structured attributes as output.  
Once trained, the networks extract private attributes from any intercepted embedding instantly.

## Two-stage attack

| Stage | Predicts                                              |
|-------|-------------------------------------------------------|
| 1     | name, relation                                        |
| 2     | object — conditioned on the predicted relation        |

**Why two stages for object?**

There are ~1 500 possible objects in the dataset, making a single classifier unstable.  
But object depends heavily on relation : *rap* only makes sense for *like\_music*, *teacher* for *has\_profession*.  
So we train one small object classifier per relation — each predicts a handful of relevant values.

## Threat model

The attacker queries a public embedding API on a large set of labeled persona sentences.  
All classifiers are trained once and applied instantly to any future stolen embedding.

In [ ]:
import warnings
import faiss
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

warnings.filterwarnings('ignore', category=ConvergenceWarning)

TRAIN_DATA_FILE        = 'data/sentences_train_text_db.parquet'
TRAIN_INDEX_FILE       = 'data/sentences_train_vector_db.index'
TARGET_TEXT_FILE       = 'data/sentences_target_text_db.parquet'
TARGET_INDEX_FILE      = 'data/sentences_target_vector_db.index'

TARGET_ID              = 'OCTO_01'
MIN_CLASS_COUNT        = 5
MIN_OBJECT_CLASS_COUNT = 3
VAL_SIZE               = 0.2
RANDOM_SEED            = 42
TOP_K                  = 5

def print_top_k(label: str, probs: np.ndarray, encoder: LabelEncoder) -> None:
    top_idx = probs.argsort()[::-1][:TOP_K]
    print(f'{label} :')
    for rank, idx in enumerate(top_idx, 1):
        print(f'{rank}. {probs[idx]:.4f} | {encoder.classes_[idx]}')
    print()

In [ ]:
df = pd.read_parquet(TRAIN_DATA_FILE)

In [ ]:
faiss_index = faiss.read_index(TRAIN_INDEX_FILE)
all_emb     = np.zeros((faiss_index.ntotal, faiss_index.d), dtype=np.float32)
faiss_index.reconstruct_n(0, faiss_index.ntotal, all_emb)

In [ ]:
name_counts     = df['name'].value_counts()
df              = df[df['name'].isin(name_counts[name_counts >= MIN_CLASS_COUNT].index)].reset_index(drop=True)
relation_counts = df['relation'].value_counts()
df              = df[df['relation'].isin(relation_counts[relation_counts >= MIN_CLASS_COUNT].index)].reset_index(drop=True)

embeddings = all_emb[df['id'].values]
del all_emb

In [ ]:
le_name     = LabelEncoder()
le_relation = LabelEncoder()

y_name     = le_name.fit_transform(df['name'])
y_relation = le_relation.fit_transform(df['relation'])

X_train, X_val, yn_train, yn_val, yr_train, yr_val, idx_train, idx_val = train_test_split(
    embeddings, y_name, y_relation, np.arange(len(df)),
    test_size=VAL_SIZE, random_state=RANDOM_SEED,
)
df_train = df.iloc[idx_train].reset_index(drop=True)

In [ ]:
mlp_name = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, random_state=RANDOM_SEED)
mlp_name.fit(X_train, yn_train)

print(f'name val accuracy : {mlp_name.score(X_val, yn_val):.2%}')

In [ ]:
mlp_relation = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200, random_state=RANDOM_SEED)
mlp_relation.fit(X_train, yr_train)

print(f'relation val accuracy : {mlp_relation.score(X_val, yr_val):.2%}')

In [ ]:
mlp_object_by_relation     = {}
object_encoder_by_relation = {}
skipped                    = []

for relation in tqdm(df_train['relation'].unique(), desc='training object classifiers'):
    mask     = df_train['relation'] == relation
    X_rel    = X_train[mask.values]
    y_obj    = df_train.loc[mask, 'object']

    counts   = y_obj.value_counts()
    keep     = counts[counts >= MIN_OBJECT_CLASS_COUNT].index
    obj_mask = y_obj.isin(keep)
    X_rel    = X_rel[obj_mask.values]
    y_obj    = y_obj[obj_mask]

    if y_obj.nunique() < 2:
        skipped.append(relation)
        continue

    le_obj = LabelEncoder()
    yo     = le_obj.fit_transform(y_obj)

    mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=100, random_state=RANDOM_SEED)
    mlp.fit(X_rel, yo)

    mlp_object_by_relation[relation]     = mlp
    object_encoder_by_relation[relation] = le_obj

In [ ]:
target_meta_df = pd.read_parquet(TARGET_TEXT_FILE)
target_fi      = faiss.read_index(TARGET_INDEX_FILE)
target_all_emb = np.zeros((target_fi.ntotal, target_fi.d), dtype=np.float32)
target_fi.reconstruct_n(0, target_fi.ntotal, target_all_emb)

target_row     = target_meta_df[target_meta_df['target_id'] == TARGET_ID].iloc[0]
target_emb_vec = target_all_emb[target_row['id']].reshape(1, -1)

In [ ]:
probs_name     = mlp_name.predict_proba(target_emb_vec)[0]
probs_relation = mlp_relation.predict_proba(target_emb_vec)[0]

print_top_k('Name',     probs_name,     le_name)
print_top_k('Relation', probs_relation, le_relation)

top_relations = le_relation.classes_[probs_relation.argsort()[::-1][:3]]
print('Objects by top predicted relations :')
for relation in top_relations:
    if relation not in mlp_object_by_relation:
        continue
    probs_obj = mlp_object_by_relation[relation].predict_proba(target_emb_vec)[0]
    le_obj    = object_encoder_by_relation[relation]
    top_obj   = probs_obj.argsort()[::-1][:3]
    print(f'relation: {relation}')
    for rank, idx in enumerate(top_obj, 1):
        print(f'{rank}. {probs_obj[idx]:.4f} | {le_obj.classes_[idx]}')
    print()

best_relation = le_relation.classes_[probs_relation.argmax()]
print('Final predicted attributes :')
print(f'name: {le_name.classes_[probs_name.argmax()]}')
print(f'relation: {best_relation}')
if best_relation in mlp_object_by_relation:
    probs_obj = mlp_object_by_relation[best_relation].predict_proba(target_emb_vec)[0]
    best_obj  = object_encoder_by_relation[best_relation].classes_[probs_obj.argmax()]
    print(f'object: {best_obj}')